# V4 Universal Football Model — Footballdata.io Ingestion

This notebook tests the new Footballdata.io API, replacing the deprecated Sofascore integration.
We will use this notebook to explore the JSON structure and build the parsing logic before wiring it into the live V4 backend.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load the API key from .env
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("FOOTBALLDATA_API_KEY")

if not API_KEY:
    print("⚠️ API Key not found! Please check your .env file.")
else:
    print(f"✅ API Key loaded: {API_KEY[:5]}...{API_KEY[-5:]}")

## 1. Basic API Fetch Function
Let's define a helper function to hit the Footballdata.io endpoints based on their documentation.

In [ ]:
# The correct base URL according to https://footballdata.io/documentation/endpoints/
BASE_URL = "https://footballdata.io/api/v1"

def fetch_footballdata(endpoint, params=None):
    """Helper to fetch data from footballdata.io"""
    if params is None:
        params = {}
        
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    
    # Standard Authorization approaches.
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status() # Raise an exception for bad status codes
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        if 'response' in locals() and response is not None:
            print(f"Response text: {response.text}")
        return None

## 2. Fetching Match Endpoints
Let's securely inspect what `fixtures/today` actually returns so we can map it.

In [ ]:
print("Fetching /fixtures/today...")
today_data = fetch_footballdata("fixtures/today")

if today_data:
    print("\n✅ Data fetched successfully! Here is the top-level structure:")
    print(f"Keys in response: {list(today_data.keys())}")
    
    if "data" in today_data:
        print(f"Type of 'data': {type(today_data['data'])}")
        
        if isinstance(today_data["data"], dict):
            print(f"Keys inside 'data': {list(today_data['data'].keys())}")
            # Print a snippet of the dictionary safely
            print("\nSnippet of data:")
            print(json.dumps(today_data["data"], indent=2)[:500])
            
        elif isinstance(today_data["data"], list) and len(today_data["data"]) > 0:
            print("\nFirst item in 'data' list:")
            print(json.dumps(today_data["data"][0], indent=2))
        else:
            print("'data' is an empty list or unrecognized format.")
else:
    print("\n⚠️ Failed to fetch /fixtures/today")